In [1]:
import pandas as pd
import numpy as np

In [2]:
m_regular_season_detailed = pd.read_csv('/Users/shaurya/Downloads/MRegularSeasonDetailedResults.csv')

In [3]:
m_seeds = pd.read_csv('/Users/shaurya/Downloads/MNCAATourneySeeds.csv')

In [4]:
m_tourney_detailed_result = pd.read_csv('/Users/shaurya/Downloads/MNCAATourneyDetailedResults.csv')

In [45]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import log_loss, accuracy_score

In [47]:
#TeamID = 1458 , we will work on '05,06,07 an pred on 08

In [48]:
TEAM_ID = 1458
YEARS   = [2005, 2006, 2007]

reg_wins = m_regular_season_detailed[
    (m_regular_season_detailed['WTeamID'] == TEAM_ID) &
    (m_regular_season_detailed['Season'].isin(YEARS))
].copy()
reg_wins['Result'] = 'Win'

reg_losses = m_regular_season_detailed[
    (m_regular_season_detailed['LTeamID'] == TEAM_ID) &
    (m_regular_season_detailed['Season'].isin(YEARS))
].copy()
reg_losses['Result'] = 'Loss'
reg_all = pd.concat([reg_wins, reg_losses]).sort_values(
    ['Season', 'DayNum']).reset_index(drop=True)

print(f"📋 REGULAR SEASON GAMES for Team {TEAM_ID} (2005–2007):")
print(f"   Wins   : {len(reg_wins)}")
print(f"   Losses : {len(reg_losses)}")
print(f"   Total  : {len(reg_all)}")
print(reg_all[['Season','DayNum','WTeamID','WScore',
               'LTeamID','LScore','Result']].to_string(index=False))


📋 REGULAR SEASON GAMES for Team 1458 (2005–2007):
   Wins   : 70
   Losses : 24
   Total  : 94
 Season  DayNum  WTeamID  WScore  LTeamID  LScore Result
   2005      19     1458      77     1335      44    Win
   2005      22     1458      72     1364      61    Win
   2005      26     1337      75     1458      61   Loss
   2005      29     1458      69     1268      64    Win
   2005      33     1458      70     1353      62    Win
   2005      36     1458      65     1453      55    Win
   2005      40     1266      63     1458      54   Loss
   2005      44     1458      66     1454      37    Win
   2005      52     1458      85     1422      53    Win
   2005      56     1458      89     1441      49    Win
   2005      58     1458      76     1104      62    Win
   2005      65     1458      77     1345      68    Win
   2005      68     1231      74     1458      61   Loss
   2005      71     1458      72     1326      66    Win
   2005      76     1458      62     1277      59 

In [51]:
tourney_wins = m_tourney_detailed_result[
    (m_tourney_detailed_result['WTeamID'] == TEAM_ID) &
    (m_tourney_detailed_result['Season'].isin(YEARS))
].copy()
tourney_wins['Result'] = 'Win'
tourney_losses = m_tourney_detailed_result[
    (m_tourney_detailed_result['LTeamID'] == TEAM_ID) &
    (m_tourney_detailed_result['Season'].isin(YEARS))
].copy()
tourney_losses['Result'] = 'Loss'
tourney_all = pd.concat([tourney_wins, tourney_losses]).sort_values(
    ['Season', 'DayNum']).reset_index(drop=True)

print(f"\n TOURNAMENT GAMES for Team {TEAM_ID} (2005–2007):")
print(f"   Wins   : {len(tourney_wins)}")
print(f"   Losses : {len(tourney_losses)}")
print(f"   Total  : {len(tourney_all)}")
print(tourney_all[['Season','DayNum','WTeamID','WScore',
                   'LTeamID','LScore','Result']].to_string(index=False))



 TOURNAMENT GAMES for Team 1458 (2005–2007):
   Wins   : 4
   Losses : 3
   Total  : 7
 Season  DayNum  WTeamID  WScore  LTeamID  LScore Result
   2005     137     1458      57     1320      52    Win
   2005     139     1458      71     1137      62    Win
   2005     144     1458      65     1301      56    Win
   2005     146     1314      88     1458      82   Loss
   2006     137     1112      94     1458      75   Loss
   2007     137     1458      76     1394      63    Win
   2007     139     1424      74     1458      68   Loss


In [53]:
seeds_team = m_seeds[
    (m_seeds['TeamID'] == TEAM_ID) &
    (m_seeds['Season'].isin(YEARS))
].copy()

if seeds_team['Seed'].dtype == object:
    seeds_team['SeedNum'] = seeds_team['Seed'].astype(str).str.extract(r'(\d+)').astype(int)
else:
    seeds_team['SeedNum'] = seeds_team['Seed'].astype(int)

print(f"\n SEEDS for Team {TEAM_ID} (2005–2007):")
print(seeds_team[['Season','Seed','SeedNum']].to_string(index=False))


 SEEDS for Team 1458 (2005–2007):
 Season  Seed  SeedNum
   2005     6        6
   2006     9        9
   2007     2        2


In [58]:
def get_team_avgs(season):
    reg = m_regular_season_detailed[
        m_regular_season_detailed['Season'] == season].copy()

    w = reg[['WTeamID','WFGM','WFGA','WFGM3','WFGA3','WFTM','WFTA',
             'WOR','WDR','WAst','WTO','WStl','WBlk','WPF','WScore']].copy()
    w.columns = ['TeamID','FGM','FGA','FGM3','FGA3','FTM','FTA',
                 'OR','DR','Ast','TO','Stl','Blk','PF','Score']

    l = reg[['LTeamID','LFGM','LFGA','LFGM3','LFGA3','LFTM','LFTA',
             'LOR','LDR','LAst','LTO','LStl','LBlk','LPF','LScore']].copy()
    l.columns = ['TeamID','FGM','FGA','FGM3','FGA3','FTM','FTA',
                 'OR','DR','Ast','TO','Stl','Blk','PF','Score']

    avgs = pd.concat([w, l]).groupby('TeamID').mean().reset_index()

    wins   = reg.groupby('WTeamID').size().reset_index(name='Wins')
    losses = reg.groupby('LTeamID').size().reset_index(name='Losses')
    avgs = avgs.merge(wins,   left_on='TeamID', right_on='WTeamID', how='left').drop('WTeamID', axis=1)
    avgs = avgs.merge(losses, left_on='TeamID', right_on='LTeamID', how='left').drop('LTeamID', axis=1)
    avgs['Wins']    = avgs['Wins'].fillna(0)
    avgs['Losses']  = avgs['Losses'].fillna(0)
    avgs['WinRate'] = avgs['Wins'] / (avgs['Wins'] + avgs['Losses'])
    avgs['TS_pct']  = avgs['Score'] / (2 * (avgs['FGA'] + 0.44 * avgs['FTA']))
    avgs['AST_TO']  = avgs['Ast']   / (avgs['TO'] + 1e-5)
    avgs['OffEff']  = (avgs['Score'] /
                      (avgs['FGA'] - avgs['OR'] + avgs['TO'] + 0.44 * avgs['FTA'] + 1e-5)) * 100
    avgs['RebRate'] = avgs['OR'] / (avgs['OR'] + avgs['DR'] + 1e-5)
    return avgs

In [59]:
print(f"\n SEASON AVERAGES for Team {TEAM_ID} (2005–2007):")

for year in YEARS:
    avgs = get_team_avgs(year)   # reusing our helper function
    row  = avgs[avgs['TeamID'] == TEAM_ID]

    if row.empty:
        print(f"\n  {year}: ❌ No data found")
    else:
        r = row.iloc[0]
        print(f"\n  {year}:")
        print(f"    Points/Game    : {r['Score']:.2f}")
        print(f"    Win Rate       : {r['WinRate']:.3f}  ({int(r['Wins'])}W - {int(r['Losses'])}L)")
        print(f"    True Shoot %   : {r['TS_pct']:.3f}")
        print(f"    Assist/TO Ratio: {r['AST_TO']:.3f}")
        print(f"    Off Efficiency : {r['OffEff']:.2f}")
        print(f"    Reb Rate       : {r['RebRate']:.3f}")
        print(f"    Assists/Game   : {r['Ast']:.2f}")
        print(f"    Turnovers/Game : {r['TO']:.2f}")
        print(f"    Steals/Game    : {r['Stl']:.2f}")
        print(f"    Blocks/Game    : {r['Blk']:.2f}")



 SEASON AVERAGES for Team 1458 (2005–2007):

  2005:
    Points/Game    : 67.17
    Win Rate       : 0.733  (22W - 8L)
    True Shoot %   : 0.542
    Assist/TO Ratio: 1.140
    Off Efficiency : 106.70
    Reb Rate       : 0.299
    Assists/Game   : 13.07
    Turnovers/Game : 11.47
    Steals/Game    : 5.47
    Blocks/Game    : 1.97

  2006:
    Points/Game    : 71.00
    Win Rate       : 0.633  (19W - 11L)
    True Shoot %   : 0.525
    Assist/TO Ratio: 1.152
    Off Efficiency : 107.02
    Reb Rate       : 0.354
    Assists/Game   : 13.40
    Turnovers/Game : 11.63
    Steals/Game    : 5.37
    Blocks/Game    : 3.37

  2007:
    Points/Game    : 70.09
    Win Rate       : 0.853  (29W - 5L)
    True Shoot %   : 0.560
    Assist/TO Ratio: 1.201
    Off Efficiency : 111.82
    Reb Rate       : 0.324
    Assists/Game   : 13.74
    Turnovers/Game : 11.44
    Steals/Game    : 5.68
    Blocks/Game    : 3.26


In [61]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import log_loss, accuracy_score

TEAM_ID      = 1458
TRAIN_YEARS  = [2005, 2006, 2007]
PREDICT_YEAR = 2008

feature_cols = ['diff_Score','diff_TS_pct','diff_AST_TO','diff_OffEff',
                'diff_RebRate','diff_OR','diff_Ast','diff_TO',
                'diff_Stl','diff_Blk','diff_WinRate','diff_Seed']

def get_seeds(season):
    s = m_seeds[m_seeds['Season'] == season].copy()
    if s['Seed'].dtype == object:
        s['SeedNum'] = s['Seed'].astype(str).str.extract(r'(\d+)').astype(int)
    else:
        s['SeedNum'] = s['Seed'].astype(int)
    return s

def build_matchup(t1, t2, avgs, seeds, label):
    s1 = avgs[avgs['TeamID'] == t1]
    s2 = avgs[avgs['TeamID'] == t2]
    if s1.empty or s2.empty:
        return None
    s1, s2 = s1.iloc[0], s2.iloc[0]
    sd1   = seeds[seeds['TeamID'] == t1]
    sd2   = seeds[seeds['TeamID'] == t2]
    seed1 = sd1['SeedNum'].values[0] if not sd1.empty else 8
    seed2 = sd2['SeedNum'].values[0] if not sd2.empty else 8
    return {
        'Team1': t1, 'Team2': t2,
        'diff_Score':   s1['Score']   - s2['Score'],
        'diff_TS_pct':  s1['TS_pct']  - s2['TS_pct'],
        'diff_AST_TO':  s1['AST_TO']  - s2['AST_TO'],
        'diff_OffEff':  s1['OffEff']  - s2['OffEff'],
        'diff_RebRate': s1['RebRate'] - s2['RebRate'],
        'diff_OR':      s1['OR']      - s2['OR'],
        'diff_Ast':     s1['Ast']     - s2['Ast'],
        'diff_TO':      s1['TO']      - s2['TO'],
        'diff_Stl':     s1['Stl']     - s2['Stl'],
        'diff_Blk':     s1['Blk']     - s2['Blk'],
        'diff_WinRate': s1['WinRate'] - s2['WinRate'],
        'diff_Seed':    seed1         - seed2,
        'label': label
    }


print(" Building training set...")
train_rows = []

for year in TRAIN_YEARS:
    avgs       = get_team_avgs(year)
    seeds_year = get_seeds(year)

    
    for _, game in m_regular_season_detailed[
            m_regular_season_detailed['Season'] == year].iterrows():
        row = build_matchup(game['WTeamID'], game['LTeamID'], avgs, seeds_year, 1)
        if row: train_rows.append(row)
        row = build_matchup(game['LTeamID'], game['WTeamID'], avgs, seeds_year, 0)
        if row: train_rows.append(row)

    
    for _, game in m_tourney_detailed_result[
            m_tourney_detailed_result['Season'] == year].iterrows():
        row = build_matchup(game['WTeamID'], game['LTeamID'], avgs, seeds_year, 1)
        if row: train_rows.append(row)
        row = build_matchup(game['LTeamID'], game['WTeamID'], avgs, seeds_year, 0)
        if row: train_rows.append(row)

    print(f"  ✅ {year} done — total rows so far: {len(train_rows)}")

df_train = pd.DataFrame(train_rows)
print(f"\n✅ Training Set: {df_train.shape[0]} rows")

print("\n🔄 Training model...")
X_train = df_train[feature_cols]
y_train = df_train['label']

scaler         = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
model          = LogisticRegression(max_iter=1000, C=1.0)

cv          = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_logloss  = cross_val_score(model, X_train_scaled, y_train, cv=cv, scoring='neg_log_loss')
cv_accuracy = cross_val_score(model, X_train_scaled, y_train, cv=cv, scoring='accuracy')

print(f"\n📈 Cross-Validated Performance:")
print(f"  Log Loss : {-cv_logloss.mean():.4f} ± {cv_logloss.std():.4f}")
print(f"  Accuracy : {cv_accuracy.mean():.4f} ± {cv_accuracy.std():.4f}")

model.fit(X_train_scaled, y_train)
print("✅ Model trained!")

print(f"\n🔄 Building 2008 test data for Team {TEAM_ID}...")
avgs_2008  = get_team_avgs(PREDICT_YEAR)
seeds_2008 = get_seeds(PREDICT_YEAR)

tourney_2008 = m_tourney_detailed_result[
    m_tourney_detailed_result['Season'] == PREDICT_YEAR]
games_team = tourney_2008[
    (tourney_2008['WTeamID'] == TEAM_ID) |
    (tourney_2008['LTeamID'] == TEAM_ID)]

print(f" Team {TEAM_ID} played {len(games_team)} tournament games in {PREDICT_YEAR}")

test_rows = []
for _, game in games_team.iterrows():
    opponent = game['LTeamID'] if game['WTeamID'] == TEAM_ID else game['WTeamID']
    label    = 1 if game['WTeamID'] == TEAM_ID else 0
    row = build_matchup(TEAM_ID, opponent, avgs_2008, seeds_2008, label)
    if row:
        row['Opponent']   = opponent
        row['Actual_Win'] = label
        test_rows.append(row)

df_test       = pd.DataFrame(test_rows)
X_test_scaled = scaler.transform(df_test[feature_cols])

df_test['Win_Probability'] = model.predict_proba(X_test_scaled)[:, 1]
df_test['Predicted_Win']   = (df_test['Win_Probability'] >= 0.5).astype(int)

print(f"\n📊 Predictions for Team {TEAM_ID} in {PREDICT_YEAR} Tournament:")
print(df_test[['Team1','Opponent','Win_Probability',
               'Predicted_Win','Actual_Win']].to_string(index=False))

correct = (df_test['Predicted_Win'] == df_test['Actual_Win']).sum()
total   = len(df_test)
print(f"\n🎯 {correct}/{total} games predicted correctly")

if total > 0:
    print(f"  Log Loss : {log_loss(df_test['Actual_Win'], df_test['Win_Probability']):.4f}")
    print(f"  Accuracy : {accuracy_score(df_test['Actual_Win'], df_test['Predicted_Win']):.4f}")

# Feature Importance
coef_df = pd.DataFrame({
    'Feature':     feature_cols,
    'Coefficient': model.coef_[0]
}).sort_values('Coefficient', ascending=False)
print("\n🔍 Feature Importances:")
print(coef_df.to_string(index=False))

🔄 Building training set...
  ✅ 2005 done — total rows so far: 9478
  ✅ 2006 done — total rows so far: 19120
  ✅ 2007 done — total rows so far: 29334

✅ Training Set: 29334 rows

🔄 Training model...

📈 Cross-Validated Performance:
  Log Loss : 0.5161 ± 0.0047
  Accuracy : 0.7387 ± 0.0067
✅ Model trained!

🔄 Building 2008 test data for Team 1458...
✅ Team 1458 played 3 tournament games in 2008

📊 Predictions for Team 1458 in 2008 Tournament:
 Team1  Opponent  Win_Probability  Predicted_Win  Actual_Win
  1458      1168         0.846534              1           1
  1458      1243         0.863785              1           1
  1458      1172         0.687363              1           0

🎯 2/3 games predicted correctly
  Log Loss : 0.4919
  Accuracy : 0.6667

🔍 Feature Importances:
     Feature  Coefficient
diff_WinRate     1.316796
 diff_OffEff     0.171407
     diff_OR     0.166394
    diff_Blk     0.128930
    diff_Ast     0.108875
 diff_TS_pct     0.083957
    diff_Stl     0.017057
 diff_A

In [62]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import log_loss, accuracy_score

# ══════════════════════════════════════════════════════════════════════
# CONFIG
# ══════════════════════════════════════════════════════════════════════
TRAIN_YEARS  = [2005, 2006, 2007]
PREDICT_YEAR = 2008

feature_cols = ['diff_Score','diff_TS_pct','diff_AST_TO','diff_OffEff',
                'diff_RebRate','diff_OR','diff_Ast','diff_TO',
                'diff_Stl','diff_Blk','diff_WinRate','diff_Seed']

# ══════════════════════════════════════════════════════════════════════
# HELPER FUNCTIONS
# ══════════════════════════════════════════════════════════════════════

def get_team_avgs(season):
    reg = m_regular_season_detailed[
        m_regular_season_detailed['Season'] == season].copy()

    w = reg[['WTeamID','WFGM','WFGA','WFGM3','WFGA3','WFTM','WFTA',
             'WOR','WDR','WAst','WTO','WStl','WBlk','WPF','WScore']].copy()
    w.columns = ['TeamID','FGM','FGA','FGM3','FGA3','FTM','FTA',
                 'OR','DR','Ast','TO','Stl','Blk','PF','Score']

    l = reg[['LTeamID','LFGM','LFGA','LFGM3','LFGA3','LFTM','LFTA',
             'LOR','LDR','LAst','LTO','LStl','LBlk','LPF','LScore']].copy()
    l.columns = ['TeamID','FGM','FGA','FGM3','FGA3','FTM','FTA',
                 'OR','DR','Ast','TO','Stl','Blk','PF','Score']

    avgs = pd.concat([w, l]).groupby('TeamID').mean().reset_index()

    wins   = reg.groupby('WTeamID').size().reset_index(name='Wins')
    losses = reg.groupby('LTeamID').size().reset_index(name='Losses')
    avgs = avgs.merge(wins,   left_on='TeamID', right_on='WTeamID', how='left').drop('WTeamID', axis=1)
    avgs = avgs.merge(losses, left_on='TeamID', right_on='LTeamID', how='left').drop('LTeamID', axis=1)
    avgs['Wins']    = avgs['Wins'].fillna(0)
    avgs['Losses']  = avgs['Losses'].fillna(0)
    avgs['WinRate'] = avgs['Wins'] / (avgs['Wins'] + avgs['Losses'])
    avgs['TS_pct']  = avgs['Score'] / (2 * (avgs['FGA'] + 0.44 * avgs['FTA']))
    avgs['AST_TO']  = avgs['Ast']   / (avgs['TO'] + 1e-5)
    avgs['OffEff']  = (avgs['Score'] /
                      (avgs['FGA'] - avgs['OR'] + avgs['TO'] + 0.44 * avgs['FTA'] + 1e-5)) * 100
    avgs['RebRate'] = avgs['OR'] / (avgs['OR'] + avgs['DR'] + 1e-5)
    return avgs


def get_seeds(season):
    s = m_seeds[m_seeds['Season'] == season].copy()
    if s['Seed'].dtype == object:
        s['SeedNum'] = s['Seed'].astype(str).str.extract(r'(\d+)').astype(int)
    else:
        s['SeedNum'] = s['Seed'].astype(int)
    return s


def build_matchup(t1, t2, avgs, seeds, label):
    s1 = avgs[avgs['TeamID'] == t1]
    s2 = avgs[avgs['TeamID'] == t2]
    if s1.empty or s2.empty:
        return None
    s1, s2 = s1.iloc[0], s2.iloc[0]

    sd1   = seeds[seeds['TeamID'] == t1]
    sd2   = seeds[seeds['TeamID'] == t2]
    seed1 = sd1['SeedNum'].values[0] if not sd1.empty else 8
    seed2 = sd2['SeedNum'].values[0] if not sd2.empty else 8

    return {
        'Team1': t1, 'Team2': t2,
        'diff_Score':   s1['Score']   - s2['Score'],
        'diff_TS_pct':  s1['TS_pct']  - s2['TS_pct'],
        'diff_AST_TO':  s1['AST_TO']  - s2['AST_TO'],
        'diff_OffEff':  s1['OffEff']  - s2['OffEff'],
        'diff_RebRate': s1['RebRate'] - s2['RebRate'],
        'diff_OR':      s1['OR']      - s2['OR'],
        'diff_Ast':     s1['Ast']     - s2['Ast'],
        'diff_TO':      s1['TO']      - s2['TO'],
        'diff_Stl':     s1['Stl']     - s2['Stl'],
        'diff_Blk':     s1['Blk']     - s2['Blk'],
        'diff_WinRate': s1['WinRate'] - s2['WinRate'],
        'diff_Seed':    seed1         - seed2,
        'label':        label
    }


# ══════════════════════════════════════════════════════════════════════
# STEP 1 — Build Training Set (2005–2007 Regular Season + Tournament)
# ══════════════════════════════════════════════════════════════════════
print("🔄 Building training set from 2005–2007...")
train_rows = []

for year in TRAIN_YEARS:
    avgs       = get_team_avgs(year)
    seeds_year = get_seeds(year)

    # Regular Season
    for _, game in m_regular_season_detailed[
            m_regular_season_detailed['Season'] == year].iterrows():
        row = build_matchup(game['WTeamID'], game['LTeamID'], avgs, seeds_year, 1)
        if row: train_rows.append(row)
        row = build_matchup(game['LTeamID'], game['WTeamID'], avgs, seeds_year, 0)
        if row: train_rows.append(row)

    # Tournament
    for _, game in m_tourney_detailed_result[
            m_tourney_detailed_result['Season'] == year].iterrows():
        row = build_matchup(game['WTeamID'], game['LTeamID'], avgs, seeds_year, 1)
        if row: train_rows.append(row)
        row = build_matchup(game['LTeamID'], game['WTeamID'], avgs, seeds_year, 0)
        if row: train_rows.append(row)

    print(f"  ✅ {year} done — total rows: {len(train_rows)}")

df_train = pd.DataFrame(train_rows)
print(f"\n✅ Training Set: {df_train.shape[0]} rows")


# ══════════════════════════════════════════════════════════════════════
# STEP 2 — Train Logistic Regression
# ══════════════════════════════════════════════════════════════════════
print("\n🔄 Training model...")

X_train        = df_train[feature_cols]
y_train        = df_train['label']
scaler         = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
model          = LogisticRegression(max_iter=1000, C=1.0)

cv          = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_logloss  = cross_val_score(model, X_train_scaled, y_train,
                               cv=cv, scoring='neg_log_loss')
cv_accuracy = cross_val_score(model, X_train_scaled, y_train,
                               cv=cv, scoring='accuracy')

print(f"\n📈 Cross-Validated Training Performance:")
print(f"  Log Loss : {-cv_logloss.mean():.4f} ± {cv_logloss.std():.4f}")
print(f"  Accuracy : {cv_accuracy.mean():.4f} ± {cv_accuracy.std():.4f}")

model.fit(X_train_scaled, y_train)
print("✅ Model trained!")


# ══════════════════════════════════════════════════════════════════════
# STEP 3 — Build 2008 Features (regular season avgs + seeds)
# ══════════════════════════════════════════════════════════════════════
print(f"\n🔄 Computing 2008 season averages & seeds...")
avgs_2008  = get_team_avgs(PREDICT_YEAR)
seeds_2008 = get_seeds(PREDICT_YEAR)
print(f"✅ Averages computed for {len(avgs_2008)} teams")
print(f"✅ Seeds loaded for {len(seeds_2008)} tournament teams")


# ══════════════════════════════════════════════════════════════════════
# STEP 4 — Predict 2008 REGULAR SEASON
# ══════════════════════════════════════════════════════════════════════
print(f"\n🔄 Predicting 2008 Regular Season games...")

reg_2008      = m_regular_season_detailed[
    m_regular_season_detailed['Season'] == PREDICT_YEAR]
reg_test_rows = []

for _, game in reg_2008.iterrows():
    row = build_matchup(game['WTeamID'], game['LTeamID'], avgs_2008, seeds_2008, 1)
    if row:
        row['Actual_Win'] = 1
        reg_test_rows.append(row)
    row = build_matchup(game['LTeamID'], game['WTeamID'], avgs_2008, seeds_2008, 0)
    if row:
        row['Actual_Win'] = 0
        reg_test_rows.append(row)

df_reg                         = pd.DataFrame(reg_test_rows)
X_reg_scaled                   = scaler.transform(df_reg[feature_cols])
df_reg['Win_Probability']      = model.predict_proba(X_reg_scaled)[:, 1]
df_reg['Predicted_Win']        = (df_reg['Win_Probability'] >= 0.5).astype(int)
df_reg['Correct']              = (df_reg['Predicted_Win'] == df_reg['Actual_Win']).astype(int)

reg_correct = df_reg['Correct'].sum()
reg_total   = len(df_reg)

print(f"\n📊 2008 Regular Season Results:")
print(f"  Total Games      : {reg_total // 2}")
print(f"  Total Rows       : {reg_total} (both perspectives)")
print(f"  Correct          : {reg_correct}/{reg_total}")
print(f"  Accuracy         : {accuracy_score(df_reg['Actual_Win'], df_reg['Predicted_Win']):.4f}")
print(f"  Log Loss         : {log_loss(df_reg['Actual_Win'], df_reg['Win_Probability']):.4f}")
print(f"\n  Full Predictions:")
print(df_reg[['Team1','Team2','Win_Probability',
              'Predicted_Win','Actual_Win','Correct']].to_string(index=False))


# ══════════════════════════════════════════════════════════════════════
# STEP 5 — Predict 2008 TOURNAMENT
# ══════════════════════════════════════════════════════════════════════
print(f"\n🔄 Predicting 2008 Tournament games...")

tourney_2008      = m_tourney_detailed_result[
    m_tourney_detailed_result['Season'] == PREDICT_YEAR]
tourney_test_rows = []

for _, game in tourney_2008.iterrows():
    row = build_matchup(game['WTeamID'], game['LTeamID'], avgs_2008, seeds_2008, 1)
    if row:
        row['Actual_Win'] = 1
        tourney_test_rows.append(row)
    row = build_matchup(game['LTeamID'], game['WTeamID'], avgs_2008, seeds_2008, 0)
    if row:
        row['Actual_Win'] = 0
        tourney_test_rows.append(row)

df_tourney                         = pd.DataFrame(tourney_test_rows)
X_tourney_scaled                   = scaler.transform(df_tourney[feature_cols])
df_tourney['Win_Probability']      = model.predict_proba(X_tourney_scaled)[:, 1]
df_tourney['Predicted_Win']        = (df_tourney['Win_Probability'] >= 0.5).astype(int)
df_tourney['Correct']              = (df_tourney['Predicted_Win'] == df_tourney['Actual_Win']).astype(int)

tourney_correct = df_tourney['Correct'].sum()
tourney_total   = len(df_tourney)

print(f"\n📊 2008 Tournament Results:")
print(f"  Total Games      : {tourney_total // 2}")
print(f"  Total Rows       : {tourney_total} (both perspectives)")
print(f"  Correct          : {tourney_correct}/{tourney_total}")
print(f"  Accuracy         : {accuracy_score(df_tourney['Actual_Win'], df_tourney['Predicted_Win']):.4f}")
print(f"  Log Loss         : {log_loss(df_tourney['Actual_Win'], df_tourney['Win_Probability']):.4f}")
print(f"\n  Full Predictions:")
print(df_tourney[['Team1','Team2','Win_Probability',
                  'Predicted_Win','Actual_Win','Correct']].to_string(index=False))


# ══════════════════════════════════════════════════════════════════════
# STEP 6 — Combined Summary
# ══════════════════════════════════════════════════════════════════════
all_actual    = pd.concat([df_reg['Actual_Win'],    df_tourney['Actual_Win']])
all_predicted = pd.concat([df_reg['Predicted_Win'], df_tourney['Predicted_Win']])
all_probs     = pd.concat([df_reg['Win_Probability'], df_tourney['Win_Probability']])

print(f"\n{'='*55}")
print(f"📌 OVERALL 2008 SUMMARY")
print(f"{'='*55}")
print(f"  Regular Season Games : {reg_total   // 2}")
print(f"  Tournament Games     : {tourney_total // 2}")
print(f"  Total Games          : {(reg_total + tourney_total) // 2}")
print(f"  Overall Accuracy     : {accuracy_score(all_actual, all_predicted):.4f}")
print(f"  Overall Log Loss     : {log_loss(all_actual, all_probs):.4f}")
print(f"  Reg Season Accuracy  : {accuracy_score(df_reg['Actual_Win'], df_reg['Predicted_Win']):.4f}")
print(f"  Tournament Accuracy  : {accuracy_score(df_tourney['Actual_Win'], df_tourney['Predicted_Win']):.4f}")

# ── Feature Importance ────────────────────────────────────────────────
coef_df = pd.DataFrame({
    'Feature':     feature_cols,
    'Coefficient': model.coef_[0]
}).sort_values('Coefficient', ascending=False)
print("\n🔍 Feature Importances:")
print(coef_df.to_string(index=False))

🔄 Building training set from 2005–2007...
  ✅ 2005 done — total rows: 9478
  ✅ 2006 done — total rows: 19120
  ✅ 2007 done — total rows: 29334

✅ Training Set: 29334 rows

🔄 Training model...

📈 Cross-Validated Training Performance:
  Log Loss : 0.5161 ± 0.0047
  Accuracy : 0.7387 ± 0.0067
✅ Model trained!

🔄 Computing 2008 season averages & seeds...
✅ Averages computed for 342 teams
✅ Seeds loaded for 65 tournament teams

🔄 Predicting 2008 Regular Season games...

📊 2008 Regular Season Results:
  Total Games      : 5163
  Total Rows       : 10326 (both perspectives)
  Correct          : 7680/10326
  Accuracy         : 0.7438
  Log Loss         : 0.5136

  Full Predictions:
 Team1  Team2  Win_Probability  Predicted_Win  Actual_Win  Correct
  1272   1404         0.972718              1           1        1
  1404   1272         0.027282              0           0        1
  1350   1263         0.888073              1           1        1
  1263   1350         0.111927              0    